Novamente, as bibliotecas importadas serão comentadas conforme o uso.

In [15]:
from ISLP import load_data
from ISLP.models import ModelSpec as MS, sklearn_sm, summarize, poly
import numpy as np
from sklearn.base import clone
import statsmodels.api as sm
from functools import partial
import pandas as pd

Inicialmente a base de dados "Portfolio" é importada da biblioteca ISLP usando a função "load_data" da mesma.

Ademais, é definido a função "alpha_func" que retorna o coeficiente $\alpha$ estimado usando a fórmula da mínima variância. A função recebe como entrada uma base de dados "D", assumindo que a mesma tenha uma coluna "X" e "Y", juntamente com uma coluna de índices, referenciando cada medição. O índice indica quais observações deverão ser consideradas para análise.

In [ ]:
Portfolio = load_data('Portfolio')

def alpha_func(D,idx):
    cov_ = np.cov(D[['X','Y']].loc[idx],rowvar=False)
    return ((cov_[1,1]-cov_[0,1])/(cov_[0,0]+cov_[1,1]-2*cov_[0,1]))

(100, 2)


Como exemplo, foi feito a aplicação da função "alpha_func", na base de dados "Portfolio" que contêm duas colunas, "X" e "Y". Como índice, foram utilizados todas as medições, que totalizam 100.

In [17]:
print(alpha_func(Portfolio,range(100)))

0.57583207459283


A fim de obter uma base de dados alternativa, com base no algoritmo de bootstrap, podemos selecionar 100 dados aleatórios da base original (o mesmo dado pode se repetir). Com uma lista de índices aleatórios, podemos aplicar como argumento "idx" e a base original "Portfolio".

In [18]:
rng = np.random.default_rng(0)
print(alpha_func(Portfolio,rng.choice(100,100,replace=True)))

0.6074452469619004


Generalizando, foi possível criar uma função que faz exatamente os algoritmos descritos anteriormente, porém unidos em uma única função. A função "boot_SE" retorna o desvio padrão, obtido calculando o coeficiente $\alpha$ usando dados conforme o algoritmo de Bootstrap.

O argumento "func" se refere a função para calcular o coeficiente que relaciona X e Y, no nosso caso $\alpha$. A entrada "D" se refere à base de dados em que se deseja aplicar o Bootstrap. "B" diz respeito ao número de dados que serão usados no algoritmo de Bootstrap (o número de replicações do algoritmo). A variável "n" se refere ao tamanho da saída, que por padrão retorna um único valor. Por fim, a variável "seed" como o prórpio nome já diz é a seed para poder replicar o mesmo resultado em momentos e máquinas diferentes.

In [19]:
def boot_SE(func,D,n=None,B=1000,seed=0):
    rng = np.random.default_rng(seed)
    first_, second_ = 0,0
    n = n or D.shape[0]
    for _ in range(B):
        idx = rng.choice(D.index,n,replace=True)
        value = func(D,idx)
        first_ += value
        second_ += value**2
    return np.sqrt(second_ / B - (first_ / B)**2)

Como exemplo, usando a função "alpha_func" descrita anteriormente, dentro da base de dados "Portfolio", "gerando" B=1000 novos dados, o erro padrão para o coeficiente $\alpha$ foi de 0,09.

In [20]:
alpha_SE = boot_SE(alpha_func,Portfolio,B=1000,seed=0)

print(alpha_SE)

0.09118176521277699


Prosseguindo, a fim de estimar a acurácia de um modelo de regressão linear usando Bootstrap, a função "boot_OLS" foi gerada. A ideia é estimar a variabilidade dos coeficientes envolvidos nos modelos de predição, por exemplo o coeficiente de reta para regressão linear.

A função "boot_OLS" a seguir, recebe como argumento da variável "model_matrix" uma função modelo, que é replicada usando a função "clone" do scikit. A variável "response" recebe o nome da coluna da variável resposta da base de dados inserida na variável "D" da função. Ademais, também é recebido na variável "idx" os índices da base "D" que serão usados para alimentar o modelo.

Por fim, a função retorna os parâmetros resposta do modelo, como o coeficiente linear e angular da regressão.

In [21]:
def boot_OLS(model_matrix,response,D,idx):
    D_ = D.loc[idx]
    Y_ = D_[response]
    X_ = clone(model_matrix).fit_transform(D_)
    return sm.OLS(Y_,X_).fit().params

Embora a função seja mais geral, "congelaremos" os dois primeiros argumentos usando a função "partial" da biblioteca functools. Ou seja, daqui para frente ficará fixo que a função modelo "model_matrix" é dado pela função "ModelSpec" da bibliteca ISLP, aplicada na coluna "horsepower", enquanto a variável resposta será sempre a coluna "mpg".

Feito isso, para aplicar agora a função "boot_OLS" (com os congelamentos feitos) é necessário chamar a função "hp_func" com os argumentos "D" e "idx" descritos anteriormente.

In [22]:
hp_func = partial(boot_OLS, MS(['horsepower']),'mpg')

Aplicando enfim a função "hp_func" na base "Auto", escolhendo 392 valores aleatórios dentre os 392 existentes na base de dados usando a função "rng.choice", foi possível obter o coeficiente linear e angular da reta, respectivamente. Importante destacar que este processo foi feito 10 vezes dentro do loop "for". Logo, o resultado é um array cujo a primeira coluna diz respeito aos coeficientes linear, enquanto a segunda diz respeito ao coeficiente angular usando o algoritmo de Bootstap em cada amostragem.

In [23]:
Auto = pd.read_csv('Auto.csv')
Auto = Auto[Auto['horsepower']!='?'].copy()
Auto['horsepower'] = Auto['horsepower'].astype(int)
Auto = Auto.reset_index()

rng = np.random.default_rng(0)
np.array([hp_func(Auto,rng.choice(392,392,replace=True)) for _ in range(10)])

array([[39.88064456, -0.1567849 ],
       [38.73298691, -0.14699495],
       [38.31734657, -0.14442683],
       [39.91446826, -0.15782234],
       [39.43349349, -0.15072702],
       [40.36629857, -0.15912217],
       [39.62334517, -0.15449117],
       [39.0580588 , -0.14952908],
       [38.66688437, -0.14521037],
       [39.64280792, -0.15555698]])

Posteriormente, inserindo a função "hp_func" na função "boot_SE" que retorna o erro padrão dos coeficientes estimados (nesse caso os coeficiente da regressão linear), aplicando na base de dados "Auto" gerando 1000 novos valores, foi obtido os seguintes valores de erro padrão para o coeficiente linear e angular da reta respoectivamente.

In [24]:
hp_se = boot_SE(hp_func,Auto,B=1000,seed=10)

display(hp_se)

intercept     0.848807
horsepower    0.007352
dtype: float64

Com o intuito de comparação, foi gerado o erro padrão novamente, usando a mesma função de treino anterior, e aplicando a função "summarize" da biblioteca ISLP, indicando como argumento "std err", para obter apenas o erro padrão como resposta. O resultado obtido aqui diverge do resultado pelo Bootstrap devido ao fato de que aqui é utilizado o pressuposto de que a regressão linear se faz correta para o conjunto dos dados (que aqui não é verdade), enquanto o bootstrap não utiliza nenhum, e é completamente livre nesse sentido.

In [25]:

hp_model = sklearn_sm(sm.OLS,MS(['horsepower']))

hp_model.fit(Auto,Auto['mpg'])

model_se = summarize(hp_model.results_)['std err']
model_se

intercept     0.717
horsepower    0.006
Name: std err, dtype: float64

Por outro lado, repetindo o mesmo processo, mas usando agora uma regressão quadrática, o erro padrão de cada coeficiente usando bootstrap foi obtido:

In [26]:
quad_model = MS([poly('horsepower',2,raw=True)])

quad_func = partial(boot_OLS,quad_model,'mpg')
boot_SE(quad_func, Auto, B=1000)

intercept                                  2.067840
poly(horsepower, degree=2, raw=True)[0]    0.033019
poly(horsepower, degree=2, raw=True)[1]    0.000120
dtype: float64

Novamente, calculando o mesmo erro padrão para os coeficientes da regressão quadrática, usando a função "summarize" podemos comparar com os resultados do bootstrap.

Aqui, como a regressão quadrática é a que melhor representa este conjunto de dados, o mesmo erro padrão para os coeficientes foram obtidos tanto para o método bootstrap quanto para o resultado via fórmula.

In [27]:
M = sm.OLS(Auto['mpg'],quad_model.fit_transform(Auto))
summarize(M.fit())['std err']

intercept                                  1.800
poly(horsepower, degree=2, raw=True)[0]    0.031
poly(horsepower, degree=2, raw=True)[1]    0.000
Name: std err, dtype: float64